In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from transformers import pipeline
import torch

/Users/anynou/Documents/mis_repos/data-projects/data_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUTA_CSV = "reseñas_unificadas.csv"
RUTA_SALIDA_RESEÑAS = "reseñas_con_sentimiento.csv"
RUTA_SALIDA_TERMINOS_POS = "terminos_positivos.csv"
RUTA_SALIDA_TERMINOS_NEG = "terminos_negativos.csv"

In [ ]:
# ---------- 1. CARGA DE DATOS ----------

df = pd.read_csv(RUTA_CSV)

# Aseguramos que existe la columna de comentarios
if "comentario" not in df.columns:
    raise ValueError("El CSV debe tener una columna llamada 'comentario'")

# Rellenar vacíos
df["comentario"] = df["comentario"].fillna("")

In [4]:
def limpiar_texto(t: str) -> str:
    if not isinstance(t, str):
        return ""
    t = t.lower()
    t = re.sub(r"http\S+", " ", t)       # quitar URLs
    t = re.sub(r"[\r\n]+", " ", t)       # saltos de línea
    t = re.sub(r"\s+", " ", t)           # espacios múltiples
    t = t.strip()
    return t

df["comentario_clean"] = df["comentario"].apply(limpiar_texto)

In [ ]:
# ---------- 2. MODELO DE SENTIMIENTO ----------

print("Cargando modelo de sentimiento ...")

sentiment_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

def obtener_estrellas(texto: str) -> float | None:
    if not texto or not isinstance(texto, str):
        return None
    # Cortamos por si el texto es muy largo
    trunc = texto[:512]
    res = sentiment_model(trunc)[0]
    label = res["label"]  # p.ej. "4 stars"
    try:
        estrellas = int(label.split()[0])
    except Exception:
        estrellas = None
    return estrellas

print("Calculando sentimiento para las reseñas...")
df["sentiment_stars"] = df["comentario_clean"].apply(obtener_estrellas)

def clasificar_sentimiento(stars):
    if stars is None:
        return "desconocido"
    if stars <= 2:
        return "negativo"
    if stars == 3:
        return "neutro"
    return "positivo"

df["sentiment_label"] = df["sentiment_stars"].apply(clasificar_sentimiento)

Cargando modelo de sentimiento (puede tardar un poco la primera vez)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6332.48it/s]


Calculando sentimiento para las reseñas...


In [ ]:
# ---------- 3. ESTADÍSTICAS BÁSICAS ----------

print("\n--- RESUMEN BÁSICO ---")
print("Número de reseñas:", len(df))

if "rating_google" in df.columns:
    print("Media rating_google (solo Google Maps, ignorando NaN):",
          df["rating_google"].dropna().mean())

print("Media sentimiento (todas las reseñas):",
      df["sentiment_stars"].dropna().mean())

print("\nDistribución de sentimiento:")
print(df["sentiment_label"].value_counts())

if "fuente" in df.columns:
    print("\nDistribución por fuente y sentimiento:")
    print(df.pivot_table(index="fuente",
                         columns="sentiment_label",
                         values="comentario",
                         aggfunc="count").fillna(0))


--- RESUMEN BÁSICO ---
Número de reseñas: 73
Media sentimiento (todas las reseñas): 3.522727272727273

Distribución de sentimiento:
sentiment_label
positivo    55
negativo    13
neutro       5
Name: count, dtype: int64

Distribución por fuente y sentimiento:
sentiment_label  negativo  neutro  positivo
fuente                                     
google_maps             7       2        41
tripadvisor             6       3        14


In [ ]:
# ---------- 4. ANÁLISIS DE TÉRMINOS ----------

# Lista simple de stopwords
stopwords_es = [
    "el", "la", "los", "las", "un", "una", "unos", "unas",
    "de", "del", "al", "a", "en", "y", "o", "u", "que",
    "con", "por", "para", "es", "son", "muy", "más",
    "menos", "no", "sí", "se", "lo", "su", "sus",
    "como", "pero", "ya", "me", "te", "le", "les",
    "mi", "tu", "su", "nuestro", "nuestra", "vosotros",
    "ellos", "ellas", "nos", "os"
]

def top_terminos(df_subset: pd.DataFrame, n_top: int = 30) -> pd.DataFrame:
    textos = df_subset["comentario_clean"].tolist()
    if not textos:
        return pd.DataFrame(columns=["term", "freq"])

    vectorizer = CountVectorizer(
        stop_words=stopwords_es,
        ngram_range=(1, 2),  # unigramas y bigramas
        min_df=1
    )
    X = vectorizer.fit_transform(textos)
    freqs = X.sum(axis=0).A1
    terms = vectorizer.get_feature_names_out()

    term_freq = pd.DataFrame({"term": terms, "freq": freqs})
    term_freq = term_freq.sort_values("freq", ascending=False).head(n_top)
    return term_freq

# Reseñas positivas y negativas según el modelo
df_pos = df[df["sentiment_label"] == "positivo"].copy()
df_neg = df[df["sentiment_label"] == "negativo"].copy()

print("\nNúmero de reseñas positivas:", len(df_pos))
print("Número de reseñas negativas:", len(df_neg))

top_pos = top_terminos(df_pos, n_top=30)
top_neg = top_terminos(df_neg, n_top=30)

print("\n--- Términos más frecuentes en reseñas POSITIVAS ---")
print(top_pos)

print("\n--- Términos más frecuentes en reseñas NEGATIVAS ---")
print(top_neg)


Número de reseñas positivas: 55
Número de reseñas negativas: 13

--- Términos más frecuentes en reseñas POSITIVAS ---
             term  freq
143       calidad     8
193        comida     8
673      servicio     8
85            bar     7
352     excelente     5
107          buen     5
737          todo     4
482          menú     4
639   restaurante     4
687         sitio     4
113         buena     4
91           bien     4
406            he     4
623    recomiendo     3
580        precio     3
26           algo     3
712         tapas     3
268        diario     3
189         comer     3
412         hemos     3
289           día     3
511         niños     3
507            ni     3
202        comido     3
309  espectacular     3
708         tanto     3
609         punto     3
317        estaba     3
321       estaban     3
121         bueno     3

--- Términos más frecuentes en reseñas NEGATIVAS ---
         term  freq
672  servicio     7
319       fue     6
745      todo     5
162

In [ ]:
# ---------- 5. GUARDAR RESULTADOS ----------

df.to_csv(RUTA_SALIDA_RESEÑAS, index=False, encoding="utf-8")
top_pos.to_csv(RUTA_SALIDA_TERMINOS_POS, index=False, encoding="utf-8")
top_neg.to_csv(RUTA_SALIDA_TERMINOS_NEG, index=False, encoding="utf-8")

print(f"\n✅ Reseñas con sentimiento guardadas en: {RUTA_SALIDA_RESEÑAS}")
print(f"✅ Términos positivos guardados en: {RUTA_SALIDA_TERMINOS_POS}")
print(f"✅ Términos negativos guardados en: {RUTA_SALIDA_TERMINOS_NEG}")
print("\nListo. Ya puedes usar estos CSV para montar gráficos o un informe.")


✅ Reseñas con sentimiento guardadas en: reseñas_con_sentimiento.csv
✅ Términos positivos guardados en: terminos_positivos.csv
✅ Términos negativos guardados en: terminos_negativos.csv

Listo. Ya puedes usar estos CSV para montar gráficos o un informe.
